# Distilling Vibration-Analysis Expertise from Big MoE Teachers into a Small Student

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winkelermatthias/MachineLearningSamples-DeepLearningforPredictiveMaintenance/blob/claude/distill-moe-vibration-analysis-xxwyye/Code/4_distill_moe_vibration_expert.ipynb)

End-to-end **knowledge distillation** pipeline for predictive maintenance / vibration diagnostics:

| Stage | What happens |
|---|---|
| **1. Synthetic data** | Massive **in-memory** generation of physically-motivated vibration **spectral patterns** (velocity spectra + high-frequency envelope spectra) with full ground-truth labels: machine info, true running speed, fault type, ISO 20816-3 severity zone, bearing condition |
| **2. Teacher labeling** | Two big open MoE teachers produce chain-of-thought diagnoses over the measurement reports: **GLM-5.2** via [NVIDIA NIM](https://build.nvidia.com/z-ai/glm-5.2) and **Kimi K3** via the [Moonshot API](https://platform.moonshot.ai/). Answers are **rejection-sampled against the simulator's ground truth** — only traces where the teacher inferred the correct speed, fault, severity, etc. survive |
| **3. Student fine-tuning** | QLoRA SFT of a small student (Qwen3-8B on an A100 / Qwen3-4B on an L4) on the accepted teacher traces |
| **4. Evaluation** | Base vs. distilled student on a held-out ground-truth set: fault accuracy, ISO-zone accuracy, bearing-condition accuracy, speed-inference error |

**The tasks the model must solve from a peak table alone** (this is the "expertise" being distilled):
- **Infer the true running speed** — the report only gives the nameplate speed *range*; the 1× shaft peak and its harmonic comb pin down the actual RPM.
- **Identify the fault** — imbalance, misalignment, looseness, bearing outer/inner race, rolling element, gear-mesh wear — by matching non-integer orders to bearing fault-frequency ratios (BPFO/BPFI/BSF/FTF) and reading envelope sidebands.
- **Identify the bearing** — in a fraction of samples the bearing designation is withheld and must be recovered from a candidate catalogue.
- **Grade severity** — ISO 20816-3 zone (A–D) from velocity RMS + the machine's zone boundaries, and a separate bearing-condition grade from envelope amplitudes.

> **Runtime**: Colab **A100 40 GB** ("big GPU") recommended; L4 works with the 4B student.
> **Cost/time knobs**: `N_TEACHER` drives API spend and wall-time (default 4 000 calls ≈ 3–5 h with both teachers running concurrently at free-tier rate limits — raise the per-teacher `concurrency` if you have paid credits; 400 is enough for a smoke run). Teacher answers, the synthetic corpus, and the model download are all cached, so interrupted runs resume where they left off. NIM has a free developer tier at [build.nvidia.com](https://build.nvidia.com).

> **Note on model names** (checked Aug 2026): `kimi-latest` was retired by Moonshot in Jan 2026 — the current flagship alias is **`kimi-k3`**. GLM-5.2 is the current Z.ai flagship on NIM (`z-ai/glm-5.2`). A verification cell below lists the live model IDs from both APIs before anything expensive runs. Both providers permit training on outputs of their open-weights flagships, but re-check the current terms for your use case.

## 0 · Setup

In [ ]:
%%capture --no-stderr
# OpenAI-compatible clients for both teachers + HF stack for the student.
!pip install -qU "openai>=1.40" "transformers>=4.46" "datasets>=2.20" "peft>=0.13" "trl>=0.12" "bitsandbytes>=0.44" accelerate scipy tqdm pandas matplotlib

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > A100 (or L4)"
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{GPU_NAME} — {VRAM_GB:.0f} GB VRAM")

# Qwen3-8B QLoRA fits comfortably in 40 GB (A100); Qwen3-4B in 24 GB (L4).
STUDENT_MODEL = "Qwen/Qwen3-8B" if VRAM_GB >= 36 else "Qwen/Qwen3-4B"
print("Student model:", STUDENT_MODEL)

In [ ]:
import json
import math
import os
import re
from getpass import getpass

import numpy as np


def _secret(name):
    """Colab secret (key icon in the sidebar), env var, or interactive prompt."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name) or getpass(f"{name}: ")


NVIDIA_API_KEY = _secret("NVIDIA_API_KEY")      # nvapi-...  free tier at build.nvidia.com
MOONSHOT_API_KEY = _secret("MOONSHOT_API_KEY")  # sk-...     platform.moonshot.ai

TEACHERS = {
    "glm-5.2": dict(
        base_url="https://integrate.api.nvidia.com/v1",  # NVIDIA NIM, OpenAI-compatible
        api_key=NVIDIA_API_KEY,
        model="z-ai/glm-5.2",
        concurrency=6,   # NIM free tier is rate-limited; raise if you have credits
    ),
    "kimi-k3": dict(
        base_url="https://api.moonshot.ai/v1",           # Moonshot, OpenAI-compatible
        api_key=MOONSHOT_API_KEY,
        model="kimi-k3",  # 'kimi-latest' was retired Jan 2026 -> K3 is the flagship
        concurrency=4,
    ),
}

SEED = 20816            # the ISO standard number, why not
N_SAMPLES = 40_000      # synthetic spectra generated in memory (cached to disk after first run)
N_TEACHER = 4_000       # samples sent to the teachers  <-- main API cost/time knob
N_EVAL = 300            # held-out ground-truth eval set (teachers never see it)
EVAL_RUN = 150          # eval prompts actually generated per model (generation is the slow part)
BLIND_FRACTION = 0.25   # fraction where the bearing designation must be identified
EPOCHS = 3
OUT_DIR = "/content/distill_pdm"
os.makedirs(OUT_DIR, exist_ok=True)

# OPTIONAL — persist model downloads + teacher answers across Colab runtimes.
# Uncomment BEFORE the model-loading cell runs; the ~16 GB student download then
# happens once per Drive, not once per runtime:
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
# OUT_DIR = "/content/drive/MyDrive/distill_pdm"; os.makedirs(OUT_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)

In [ ]:
# Verify the teacher model IDs are live before spending anything, then smoke-test one call each.
from openai import OpenAI

for name, cfg in TEACHERS.items():
    client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"], timeout=120)
    try:
        ids = [m.id for m in client.models.list()]
        near = [i for i in ids if any(k in i.lower() for k in ("glm", "kimi"))][:12]
        status = "OK" if cfg["model"] in ids else "NOT FOUND — pick one of the nearby ids and update TEACHERS"
        print(f"[{name}] {cfg['model']}: {status}\n         nearby: {near}")
    except Exception as e:
        print(f"[{name}] models.list unavailable ({type(e).__name__}) — not fatal on some gateways")
    try:
        r = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": "Reply with the single word: pong"}],
            max_tokens=512, temperature=0,
        )
        msg = r.choices[0].message
        print(f"[{name}] smoke test → {(msg.content or '').strip()[:60]!r}")
    except Exception as e:
        print(f"[{name}] smoke test FAILED: {e}")

## 1 · Massive in-memory synthetic generation of spectral patterns + labels

We synthesize two sensor channels per sample, both at 25.6 kHz / 65 536 points (Δf ≈ 0.39 Hz):

- a **velocity** waveform (mm/s) carrying the low-frequency machine signature — 1× and harmonics, 2× misalignment, looseness combs, gear-mesh + sidebands — from which we take the 10–1000 Hz spectrum and overall RMS;
- an **acceleration** waveform (g) carrying high-frequency bearing impact bursts (repetition at BPFO/BPFI/2×BSF, resonance ring-down, amplitude modulation at 1× for inner race and FTF for rolling-element), from which we take the **envelope spectrum** (2–8 kHz demodulation band) — the classic bearing-diagnostics tool.

Bearing fault frequencies come from real kinematics: with `n` rolling elements, ball diameter `d`, pitch diameter `D`, contact angle φ

$$\mathrm{BPFO}=\tfrac{n}{2}\big(1-\tfrac{d}{D}\cos\varphi\big)\quad \mathrm{BPFI}=\tfrac{n}{2}\big(1+\tfrac{d}{D}\cos\varphi\big)\quad \mathrm{BSF}=\tfrac{D}{2d}\big(1-(\tfrac{d}{D}\cos\varphi)^2\big)\quad \mathrm{FTF}=\tfrac12\big(1-\tfrac{d}{D}\cos\varphi\big)$$

(all in orders of shaft speed). Severity labels are **measured, not asserted**: the ISO 20816-3 zone is computed from the *actual* velocity RMS of the synthesized signal against the machine's zone boundaries, and the bearing-condition grade from the *actual* largest envelope peak — so labels are self-consistent with what the model sees, by construction.

Raw waveforms are discarded after feature extraction — only compact peak tables + labels are kept, so 20 000 samples fit in a few tens of MB of RAM.

In [ ]:
from scipy import signal as sps

# ---- bearing catalogue (n rolling elements, ball dia d [mm], pitch dia D [mm], contact angle [deg])
BEARINGS = {
    "6205":   dict(n=9,  d=7.94,  D=39.04, phi=0.0),
    "6309":   dict(n=8,  d=17.46, D=72.50, phi=0.0),
    "6311":   dict(n=8,  d=20.64, D=85.50, phi=0.0),
    "7310B":  dict(n=12, d=19.05, D=81.50, phi=40.0),
    "NU216":  dict(n=16, d=15.00, D=120.0, phi=0.0),
    "22216E": dict(n=19, d=15.90, D=110.0, phi=9.0),
}


def _ratios(b):
    r = b["d"] / b["D"] * math.cos(math.radians(b["phi"]))
    return {
        "BPFO": b["n"] / 2 * (1 - r),
        "BPFI": b["n"] / 2 * (1 + r),
        "BSF":  b["D"] / (2 * b["d"]) * (1 - r**2),
        "FTF":  0.5 * (1 - r),
    }


BEARING_RATIOS = {k: _ratios(v) for k, v in BEARINGS.items()}

# ---- ISO 20816-3 zone boundaries (velocity mm/s RMS): (A/B, B/C, C/D)
ISO_BOUNDS = {
    (1, "rigid"): (2.3, 4.5, 7.1), (1, "flexible"): (3.5, 7.1, 11.0),   # group 1: > 300 kW
    (2, "rigid"): (1.4, 2.8, 4.5), (2, "flexible"): (2.3, 4.5, 7.1),    # group 2: 15–300 kW
}


def iso_bounds(power_kw, mount):
    return ISO_BOUNDS[(1 if power_kw > 300 else 2, mount)]


def iso_zone(v_rms, bounds):
    a, b, c = bounds
    return "A" if v_rms <= a else "B" if v_rms <= b else "C" if v_rms <= c else "D"


# ---- machine fleet templates
MACHINES = [
    dict(kind="end-suction centrifugal pump, direct coupled", tag="P", power=(15, 250),
         speeds=[(2940, 2985), (1465, 1492)], mount="rigid", bearings=["6309", "6311"], gear=None),
    dict(kind="centrifugal fan, belt driven", tag="FN", power=(11, 90),
         speeds=[(850, 1750)], mount="flexible", bearings=["22216E", "6309"], gear=None),
    dict(kind="oil-flooded screw compressor, direct coupled", tag="K", power=(75, 315),
         speeds=[(2950, 2978)], mount="rigid", bearings=["7310B", "NU216"], gear=None),
    dict(kind="cooling tower fan, gearbox driven (measurement at gearbox input)", tag="CT", power=(30, 132),
         speeds=[(1470, 1488)], mount="flexible", bearings=["22216E"], gear=dict(z1=17, z2=79)),
    dict(kind="process pump on VFD", tag="P", power=(22, 110),
         speeds=[(900, 2990)], mount="rigid", bearings=["6205", "6309"], gear=None),
    dict(kind="mill drive through gearbox (measurement at input pinion)", tag="M", power=(250, 900),
         speeds=[(984, 996)], mount="rigid", bearings=["NU216", "22216E"], gear=dict(z1=23, z2=104)),
]

FAULTS = ["healthy", "imbalance", "misalignment", "looseness",
          "bearing_outer_race", "bearing_inner_race", "bearing_rolling_element", "gear_mesh_wear"]

# bearing-condition grade from the largest envelope-spectrum peak (g):
COND_BOUNDS = (0.05, 0.30, 1.00)   # none < 0.05 <= early < 0.30 <= moderate < 1.00 <= severe


def cond_grade(env_pk):
    a, b, c = COND_BOUNDS
    return "none" if env_pk < a else "early" if env_pk < b else "moderate" if env_pk < c else "severe"


def sample_machine(rng):
    tpl = MACHINES[int(rng.integers(len(MACHINES)))]
    lo, hi = tpl["speeds"][int(rng.integers(len(tpl["speeds"])))]
    power = float(rng.uniform(*tpl["power"]))
    return dict(
        kind=tpl["kind"],
        asset=f"{tpl['tag']}-{int(rng.integers(100, 999))}",
        power_kW=power,
        mount=tpl["mount"],
        speed_lo=lo, speed_hi=hi,
        rpm=float(rng.uniform(lo, hi)),
        bearing=tpl["bearings"][int(rng.integers(len(tpl["bearings"])))],
        gear=tpl["gear"],
        bounds=iso_bounds(power, tpl["mount"]),
    )

In [ ]:
# ---- waveform synthesis --------------------------------------------------
FS = 25_600
N = 1 << 16                      # 65 536 samples = 2.56 s  ->  df = 0.39 Hz
T = np.arange(N) / FS


def _tone(rng, f, amp_rms):
    return amp_rms * np.sqrt(2) * np.sin(2 * np.pi * f * T + rng.uniform(0, 2 * np.pi))


def _impact_train(rng, f_imp, amp, fc, zeta=0.04, mod_f=None, mod_depth=0.0):
    """Repetitive impacts at f_imp Hz exciting a structural resonance at fc Hz."""
    n_imp = int(f_imp * N / FS) + 2
    t_imp = (np.arange(n_imp) + rng.uniform(0.0, 1.0)) / f_imp
    t_imp = t_imp + rng.normal(0.0, 0.01 / f_imp, n_imp)          # ~1 % slip jitter
    t_imp = t_imp[(t_imp >= 0) & (t_imp < N / FS)]
    w = rng.uniform(0.7, 1.3, t_imp.size)
    if mod_f:                                                     # load-zone modulation
        w = w * (1.0 + mod_depth * np.sin(2 * np.pi * mod_f * t_imp))
    imp = np.zeros(N)
    np.add.at(imp, np.round(t_imp * FS).astype(int) % N, amp * np.abs(w))
    k = np.arange(int(FS * 0.004))                                # 4 ms ring-down kernel
    kern = np.exp(-2 * np.pi * fc * zeta * k / FS) * np.sin(2 * np.pi * fc * k / FS)
    return np.convolve(imp, kern)[:N]


_SOS_V = sps.butter(4, [10, 1000], btype="bandpass", fs=FS, output="sos")
_SOS_A = sps.butter(4, [2000, 8000], btype="bandpass", fs=FS, output="sos")
_WIN = np.hanning(N)
_FAX = np.fft.rfftfreq(N, 1 / FS)


def _amp_spectrum(x):
    return np.abs(np.fft.rfft(x * _WIN)) * 2 / _WIN.sum()         # peak-amplitude spectrum


def _peaks(f, X, fmin, fmax, top):
    m = (f >= fmin) & (f <= fmax)
    fi, Xi = f[m], X[m]
    pk, _ = sps.find_peaks(Xi, prominence=Xi.max() * 0.02)
    if pk.size == 0:
        return []
    sel = np.sort(pk[np.argsort(Xi[pk])[::-1][:top]])
    return [(float(fi[i]), float(Xi[i])) for i in sel]


def synth_sample(rng, keep_wave=False):
    m = sample_machine(rng)
    fr = m["rpm"] / 60.0
    pool = [f for f in FAULTS if f != "gear_mesh_wear" or m["gear"]]
    fault = pool[int(rng.integers(len(pool)))]
    sev = 0.0 if fault == "healthy" else float(rng.uniform(0.4, 4.2))
    R = BEARING_RATIOS[m["bearing"]]
    kz = m["bounds"][2] / 4.5     # scale severity to THIS machine's zone widths

    # baseline residual imbalance/misalignment + broadband floors
    v = _tone(rng, fr, rng.uniform(0.25, 0.55)) + _tone(rng, 2 * fr, rng.uniform(0.06, 0.18))
    v = v + rng.normal(0, 0.05, N)                                # mm/s
    a = rng.normal(0, 0.02, N)                                    # g
    fc = rng.uniform(2500, 5500)                                  # bearing housing resonance

    if fault == "imbalance":
        v = v + _tone(rng, fr, 1.4 * sev * kz)
    elif fault == "misalignment":
        v = v + _tone(rng, 2 * fr, 1.1 * sev * kz) + _tone(rng, fr, 0.4 * sev * kz) + _tone(rng, 3 * fr, 0.35 * sev * kz)
    elif fault == "looseness":
        for k in range(1, 9):
            v = v + _tone(rng, k * fr, 0.6 * sev * kz / k**0.7)
        v = v + _tone(rng, 0.5 * fr, 0.2 * sev * kz)
    elif fault == "bearing_outer_race":
        f0 = R["BPFO"] * fr
        a = a + _impact_train(rng, f0, 2.0 * sev, fc)
        v = v + _tone(rng, fr, 0.2 * sev * kz)        # late-stage wear lifts 1x too
        for k in (1, 2, 3):
            v = v + _tone(rng, k * f0, 0.15 * sev * kz)
    elif fault == "bearing_inner_race":
        f0 = R["BPFI"] * fr
        a = a + _impact_train(rng, f0, 2.0 * sev, fc, mod_f=fr, mod_depth=0.8)
        v = v + _tone(rng, fr, 0.2 * sev * kz)
        for k in (1, 2):
            v = v + _tone(rng, k * f0, 0.12 * sev * kz) + _tone(rng, k * f0 + fr, 0.06 * sev * kz) + _tone(rng, k * f0 - fr, 0.06 * sev * kz)
    elif fault == "bearing_rolling_element":
        f0 = 2 * R["BSF"] * fr
        a = a + _impact_train(rng, f0, 1.8 * sev, fc, mod_f=R["FTF"] * fr, mod_depth=0.9)
        v = v + _tone(rng, fr, 0.2 * sev * kz) + _tone(rng, f0, 0.12 * sev * kz)
    elif fault == "gear_mesh_wear":
        gmf = m["gear"]["z1"] * fr
        v = v + _tone(rng, gmf, 0.7 * sev * kz) + _tone(rng, 2 * gmf, 0.2 * sev * kz)
        v = v + _tone(rng, gmf - fr, 0.25 * sev * kz) + _tone(rng, gmf + fr, 0.25 * sev * kz)

    # measured features -> labels are self-consistent by construction
    v_rms = float(np.std(sps.sosfiltfilt(_SOS_V, v)))
    a_hf = sps.sosfiltfilt(_SOS_A, a)
    crest = float(np.max(np.abs(a_hf)) / (np.std(a_hf) + 1e-12))
    env = np.abs(sps.hilbert(a_hf))
    env = env - env.mean()
    peaks_v = _peaks(_FAX, _amp_spectrum(v), 2.0, 1200.0, top=15)
    peaks_e = _peaks(_FAX, _amp_spectrum(env), 2.0, 600.0, top=10)
    env_pk = max((amp for _, amp in peaks_e), default=0.0)

    s = dict(
        machine=m, fault=fault, sev=sev, rpm=m["rpm"],
        v_rms=v_rms, crest=crest, env_pk=env_pk,
        zone=iso_zone(v_rms, m["bounds"]), cond=cond_grade(env_pk),
        peaks_v=peaks_v, peaks_e=peaks_e,
        blind=bool(rng.uniform() < BLIND_FRACTION),
    )
    if keep_wave:
        s["wave"] = dict(v=v, env=env)
    return s

In [ ]:
def render_report(s):
    """The measurement report a technician would hand to an analyst.
    Contains everything needed to solve the case — and never the answer."""
    m = s["machine"]
    L = []
    L.append("=== VIBRATION MEASUREMENT REPORT ===")
    L.append(f"Asset {m['asset']}: {m['kind']}")
    L.append(f"Rated power: {m['power_kW']:.0f} kW | mounting: {m['mount']} | line frequency: 50 Hz")
    L.append(f"Nameplate speed range: {m['speed_lo']:.0f}-{m['speed_hi']:.0f} RPM "
             "(actual running speed NOT recorded — no tacho fitted)")
    if m["gear"]:
        L.append(f"Gearbox: pinion {m['gear']['z1']} teeth / wheel {m['gear']['z2']} teeth")
    if s["blind"]:
        L.append("Bearing designation: UNKNOWN. Candidate catalogue "
                 "(fault-frequency ratios in orders of shaft speed):")
        for name, r in BEARING_RATIOS.items():
            L.append(f"    {name:8s} BPFO={r['BPFO']:.3f}  BPFI={r['BPFI']:.3f}  "
                     f"BSF={r['BSF']:.3f}  FTF={r['FTF']:.3f}")
    else:
        r = BEARING_RATIOS[m["bearing"]]
        L.append(f"Bearing: {m['bearing']} (BPFO={r['BPFO']:.3f}, BPFI={r['BPFI']:.3f}, "
                 f"BSF={r['BSF']:.3f}, FTF={r['FTF']:.3f} x shaft speed)")
    a, b, c = m["bounds"]
    L.append(f"ISO 20816-3 zone boundaries for this machine (velocity mm/s RMS): "
             f"A <= {a} < B <= {b} < C <= {c} < D")
    L.append(f"Overall velocity 10-1000 Hz: {s['v_rms']:.2f} mm/s RMS | "
             f"HF acceleration crest factor (2-8 kHz): {s['crest']:.1f}")
    L.append("--- Velocity spectrum peaks (Hz | mm/s peak) ---")
    for f, amp in s["peaks_v"]:
        L.append(f"    {f:8.2f}  |  {amp:6.3f}")
    L.append("--- Envelope spectrum peaks, 2-8 kHz demod band (Hz | g peak) ---")
    if s["peaks_e"]:
        for f, amp in s["peaks_e"]:
            L.append(f"    {f:8.2f}  |  {amp:6.3f}")
    else:
        L.append("    (no significant peaks)")
    return "\n".join(L)

In [ ]:
# ---- generate the corpus (all in memory; waveforms are dropped on the fly) ----
# The corpus is cached to disk: a re-run in the same (or Drive-backed) OUT_DIR
# reloads it in seconds instead of resynthesizing, which also keeps sample ids
# stable so previously cached teacher answers stay valid. Delete the file to
# force regeneration (e.g. after changing N_SAMPLES or the simulator).
from tqdm.auto import tqdm

CORPUS_PATH = f"{OUT_DIR}/synthetic_corpus.jsonl"
dataset = []
if os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH) as fh:
        dataset = [json.loads(l) for l in fh if l.strip()]
    if len(dataset) == N_SAMPLES:
        print(f"reloaded {len(dataset):,} cached samples from {CORPUS_PATH}")
    else:
        print(f"cache has {len(dataset):,} samples but N_SAMPLES={N_SAMPLES:,} — regenerating")
        dataset = []
if not dataset:
    for i in tqdm(range(N_SAMPLES), desc="synthesizing"):
        s = synth_sample(rng)
        s["id"] = i
        s["report"] = render_report(s)
        dataset.append(s)
    with open(CORPUS_PATH, "w") as fh:
        for s in dataset:
            fh.write(json.dumps(s) + "\n")

eval_set = dataset[:N_EVAL]                       # ground-truth holdout, teachers never see it
teacher_pool = dataset[N_EVAL:N_EVAL + N_TEACHER]
print(f"{len(dataset):,} samples | teacher pool {len(teacher_pool):,} | eval holdout {len(eval_set):,}")
print("\nExample report:\n")
print(dataset[3]["report"])
print("\nGround truth:", {k: dataset[3][k] for k in ("fault", "zone", "cond")},
      f"rpm={dataset[3]['rpm']:.0f}")

In [ ]:
# ---- sanity look at the corpus -------------------------------------------
import matplotlib.pyplot as plt
from collections import Counter

PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # fixed categorical order
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781", "text.color": "#0b0b0b",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "lines.linewidth": 2.0, "figure.dpi": 110,
})

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, key, title in ((axes[0], "fault", "Fault distribution"),
                       (axes[1], "zone", "ISO 20816-3 zone distribution")):
    cnt = Counter(s[key] for s in dataset)
    ks = sorted(cnt, key=cnt.get, reverse=True)
    ax.barh(range(len(ks)), [cnt[k] for k in ks], color=PAL[0], height=0.62)
    ax.set_yticks(range(len(ks)), ks)
    ax.invert_yaxis()
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

In [ ]:
# ---- what the signals look like ------------------------------------------
demo_rng = np.random.default_rng(7)
demo = None
while demo is None or demo["fault"] != "bearing_inner_race":
    demo = synth_sample(demo_rng, keep_wave=True)

fv, Xv = _FAX, _amp_spectrum(demo["wave"]["v"])
fe, Xe = _FAX, _amp_spectrum(demo["wave"]["env"])
fr = demo["rpm"] / 60
bpfi = BEARING_RATIOS[demo["machine"]["bearing"]]["BPFI"] * fr

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(fv[fv <= 1200], Xv[fv <= 1200], color=PAL[0])
axes[0].set_title(f"Velocity spectrum — {demo['fault']} @ {demo['rpm']:.0f} RPM", loc="left")
axes[0].set_xlabel("Hz"); axes[0].set_ylabel("mm/s peak")
axes[1].plot(fe[fe <= 400], Xe[fe <= 400], color=PAL[1])
axes[1].axvline(bpfi, color="#898781", linestyle=":", linewidth=1.2)
axes[1].annotate(f"BPFI = {bpfi:.1f} Hz", (bpfi, Xe[fe <= 400].max() * 0.92),
                 textcoords="offset points", xytext=(6, 0), color="#52514e", fontsize=9)
axes[1].set_title("Envelope spectrum (2-8 kHz demod)", loc="left")
axes[1].set_xlabel("Hz"); axes[1].set_ylabel("g peak")
fig.tight_layout()
plt.show()
print("Truth:", {k: demo[k] for k in ("fault", "zone", "cond")}, f"| 1x = {fr:.2f} Hz")

## 2 · Teacher labeling — GLM-5.2 (NVIDIA NIM) + Kimi K3 (Moonshot), rejection-sampled

Each teacher gets the raw measurement report and must produce a full chain-of-thought diagnosis ending in a strict JSON verdict. Because the simulator knows the ground truth, we **verify every trace** — inferred speed within ±4 %, exact fault type, exact ISO zone, exact bearing-condition grade, and (for blind samples) the correct bearing designation. Only fully-correct traces enter the student's training set. This is the step that turns "big model output" into "distilled expertise" — wrong reasoning never reaches the student.

Both teachers run **concurrently** with per-provider rate limits, retries with exponential backoff, and JSONL checkpointing — you can interrupt and re-run the cell; finished calls are never repeated.

In [ ]:
FAULT_LIST = ", ".join(FAULTS)
_CB = COND_BOUNDS

SYSTEM_TEACHER = """You are a senior vibration analyst (ISO 18436-2 Category IV) doing condition
monitoring for rotating machinery. You will receive one measurement report: machine nameplate
data, ISO 20816-3 zone boundaries, overall levels, a velocity-spectrum peak table and an
envelope-spectrum peak table.

Work the case step by step, exactly in this order:
1. INFER THE TRUE RUNNING SPEED. The report only gives the nameplate range. Find the 1x shaft
   peak: it must lie inside the nameplate range and be consistent with the harmonic comb
   (2x, 3x, ...) and any sideband spacings. Report speed in RPM.
2. IDENTIFY THE FAULT — exactly one of: <FAULT_LIST>.
   Convert candidate fault frequencies (ratio x shaft speed) and compare against the peak
   tables: dominant 1x -> imbalance; 2x >= 1x (with 1x, 3x) -> misalignment; long integer
   harmonic comb + 0.5x subharmonic -> looseness; envelope peaks at BPFO harmonics without
   1x sidebands -> outer race; at BPFI with +-1x sidebands -> inner race; at 2xBSF with FTF
   sidebands -> rolling element; gear-mesh frequency (teeth x 1x) with +-1x sidebands -> gear
   mesh wear; nothing beyond baseline -> healthy.
3. IF THE BEARING DESIGNATION IS UNKNOWN, identify the most likely bearing from the candidate
   catalogue by matching envelope peaks to ratio x inferred speed. Otherwise repeat the given
   designation.
4. GRADE THE ISO ZONE (A, B, C or D) from the reported overall velocity RMS against the zone
   boundaries stated in the report.
5. GRADE THE BEARING CONDITION from the largest envelope-spectrum peak:
   none < <C0> g <= early < <C1> g <= moderate < <C2> g <= severe.
6. RECOMMEND ACTIONS proportionate to the findings.

Write your full reasoning under a heading 'Analysis:' (show the arithmetic — expected fault
frequencies in Hz, which peaks matched). Then output ONE fenced json block, nothing after it:

```json
{"inferred_speed_rpm": <number>, "fault_type": "<one of the list>", "iso_zone": "A|B|C|D",
 "bearing_condition": "none|early|moderate|severe", "bearing_designation": "<name>",
 "key_evidence": ["...", "..."], "recommended_actions": ["...", "..."]}
```"""
SYSTEM_TEACHER = (SYSTEM_TEACHER
                  .replace("<FAULT_LIST>", FAULT_LIST)
                  .replace("<C0>", str(_CB[0])).replace("<C1>", str(_CB[1])).replace("<C2>", str(_CB[2])))
print(SYSTEM_TEACHER[:600], "...")

In [ ]:
import asyncio
from openai import AsyncOpenAI


async def _call_one(client, model, sem, sample, temperature=0.4, max_tokens=4096):
    async with sem:
        last_err = "empty response"
        for attempt in range(5):
            try:
                r = await client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": SYSTEM_TEACHER},
                              {"role": "user", "content": sample["report"]}],
                    temperature=temperature, max_tokens=max_tokens,
                )
                msg = r.choices[0].message
                # reasoning models may split output; prefer visible content
                text = msg.content or getattr(msg, "reasoning_content", None) or ""
                if text.strip():
                    return sample["id"], text, None
            except Exception as e:
                last_err = f"{type(e).__name__}: {e}"
                await asyncio.sleep(2 ** attempt + float(np.random.uniform(0, 1)))
        return sample["id"], None, last_err


async def run_teacher(name, cfg, samples):
    path = f"{OUT_DIR}/teacher_{name}.jsonl"
    done = set()
    if os.path.exists(path):
        with open(path) as fh:
            done = {json.loads(l)["id"] for l in fh if l.strip()}
    todo = [s for s in samples if s["id"] not in done]
    print(f"[{name}] {len(done)} cached, {len(todo)} to go")
    if not todo:
        return
    client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"], timeout=600)
    sem = asyncio.Semaphore(cfg["concurrency"])
    coros = [_call_one(client, cfg["model"], sem, s) for s in todo]
    ok, fails = 0, []
    with open(path, "a") as fh:
        for fut in tqdm(asyncio.as_completed(coros), total=len(coros), desc=name):
            sid, text, err = await fut
            if text:
                fh.write(json.dumps({"id": sid, "teacher": name, "text": text}) + "\n")
                fh.flush()
                ok += 1
            else:
                fails.append(err)
    print(f"[{name}] wrote {ok}, FAILED {len(fails)}"
          + (f" — last error: {fails[-1]}" if fails else ""))
    if fails and not ok:
        print(f"[{name}] every call failed — check the API key and model id before re-running")


# split the pool between the two teachers, run both providers concurrently
half = len(teacher_pool) // 2
assignments = {"glm-5.2": teacher_pool[:half], "kimi-k3": teacher_pool[half:]}
await asyncio.gather(*(run_teacher(n, TEACHERS[n], assignments[n]) for n in TEACHERS))
print("teacher generation done")

In [ ]:
# ---- verify against ground truth (rejection sampling) --------------------
import pandas as pd

_JSON_RE = re.compile(r"```json\s*(\{.*?\})\s*```", re.DOTALL)


def parse_answer(text):
    m = _JSON_RE.findall(text or "")
    raw = m[-1] if m else None
    if raw is None:                                   # fall back: last {...} blob
        b = re.findall(r"\{[^{}]*\}", text or "", re.DOTALL)
        raw = b[-1] if b else None
    if raw is None:
        return None
    try:
        return json.loads(raw)
    except Exception:
        return None


def check(sample, ans):
    out = dict(parsed=ans is not None, speed=False, fault=False, zone=False,
               cond=False, brg=True, accepted=False)
    if ans is None:
        out["brg"] = False
        return out
    try:
        out["speed"] = abs(float(ans.get("inferred_speed_rpm", 0)) - sample["rpm"]) / sample["rpm"] <= 0.04
    except (TypeError, ValueError):
        pass
    out["fault"] = ans.get("fault_type") == sample["fault"]
    out["zone"] = ans.get("iso_zone") == sample["zone"]
    out["cond"] = ans.get("bearing_condition") == sample["cond"]
    if sample["blind"]:
        out["brg"] = str(ans.get("bearing_designation", "")).replace(" ", "").upper() == sample["machine"]["bearing"]
    out["accepted"] = all(v for k, v in out.items() if k != "accepted")
    return out


by_id = {s["id"]: s for s in dataset}
accepted, rows = [], []
for name in TEACHERS:
    path = f"{OUT_DIR}/teacher_{name}.jsonl"
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print(f"WARNING: no answers on disk for teacher '{name}' — all its calls failed; "
              "check the key/model id and re-run the generation cell (finished calls are cached)")
        continue
    with open(path) as fh:
        for line in fh:
            rec = json.loads(line)
            s = by_id[rec["id"]]
            res = check(s, parse_answer(rec["text"]))
            rows.append({"teacher": name, **res})
            if res["accepted"]:
                accepted.append({"sample": s, "text": rec["text"], "teacher": name})

stats = pd.DataFrame(rows).groupby("teacher").mean(numeric_only=True).round(3)
print(stats)
print(f"\naccepted traces: {len(accepted)} "
      f"({len(accepted) / max(len(rows), 1):.0%} of {len(rows)} teacher answers)")
if len(accepted) < 300:
    print(f"\nWARNING: {len(accepted)} accepted traces is far too few to distill anything — "
          "the student will just memorize formatting. Raise N_TEACHER (>= 400 for a demo, "
          "1500+ for real transfer) and re-run the teacher cell; cached answers are kept.")

## 3 · Distill into the student — QLoRA SFT

The accepted teacher traces (reasoning + JSON verdict) become chat-format training examples. The student sees the same measurement reports and learns to reproduce the *verified* expert reasoning. 4-bit NF4 quantized base + LoRA adapters — comfortable on an A100, workable on an L4 with the 4B student.

In [ ]:
from datasets import Dataset

# The student prompt must carry the answer vocabulary — otherwise the base model
# answers in free text ("inner race bearing fault") and the exact-match scoring
# reads as ~0 even when the diagnosis is right.
STUDENT_SYSTEM = ("You are a vibration analysis and predictive-maintenance expert. "
                  "Analyse the measurement report step by step under a heading 'Analysis:': "
                  "infer the true running speed from the spectrum, identify the fault "
                  "(exactly one of: <FAULT_LIST>), the bearing designation if unknown, the "
                  "ISO 20816-3 zone (A/B/C/D) from the boundaries stated in the report, and "
                  "the bearing condition from the largest envelope-spectrum peak "
                  "(none < <C0> g <= early < <C1> g <= moderate < <C2> g <= severe). "
                  "Then output one fenced json block with keys: inferred_speed_rpm, fault_type, "
                  "iso_zone, bearing_condition, bearing_designation, key_evidence, "
                  "recommended_actions.")
STUDENT_SYSTEM = (STUDENT_SYSTEM
                  .replace("<FAULT_LIST>", FAULT_LIST)
                  .replace("<C0>", str(_CB[0])).replace("<C1>", str(_CB[1])).replace("<C2>", str(_CB[2])))

records = [{"messages": [
    {"role": "system", "content": STUDENT_SYSTEM},
    {"role": "user", "content": a["sample"]["report"]},
    {"role": "assistant", "content": a["text"]},
]} for a in accepted]

ds = Dataset.from_list(records).shuffle(seed=SEED).train_test_split(test_size=0.03, seed=SEED)
print(ds)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Skip the (slow) download/load on re-runs in the same session — restart the
# runtime if you actually want a fresh, un-finetuned base model back.
if "model" in globals():
    print("model already in memory — skipping reload")
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL, quantization_config=bnb, device_map="auto",
        torch_dtype=torch.bfloat16, attn_implementation="sdpa",
    )
    model.config.use_cache = True
    print(f"{STUDENT_MODEL} loaded — {sum(p.numel() for p in model.parameters()) / 1e9:.1f} B params (4-bit)")

In [ ]:
# ---- baseline: how good is the untuned student? ---------------------------
def generate_batch(mdl, prompts, bs=8, max_new_tokens=1200):
    outs = []
    mdl.eval()
    for i in tqdm(range(0, len(prompts), bs), desc="generating"):
        chats = [tok.apply_chat_template(
            [{"role": "system", "content": STUDENT_SYSTEM}, {"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        ) for p in prompts[i:i + bs]]
        enc = tok(chats, return_tensors="pt", padding=True).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tok.pad_token_id or tok.eos_token_id, use_cache=True)
        outs += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


def score(samples, outputs):
    """Accuracy over ALL samples — an unparseable answer counts as wrong."""
    res = [check(s, parse_answer(o)) for s, o in zip(samples, outputs)]
    df = pd.DataFrame(res)
    return {"parse": df["parsed"].mean(), "fault": df["fault"].mean(), "zone": df["zone"].mean(),
            "cond": df["cond"].mean(), "speed<=4%": df["speed"].mean()}


eval_run = eval_set[:EVAL_RUN]
base_out = generate_batch(model, [s["report"] for s in eval_run])
base_scores = score(eval_run, base_out)
print("base student:", {k: f"{v:.2f}" for k, v in base_scores.items()})

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_cfg = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
BS = 4 if VRAM_GB >= 36 else 2          # effective batch stays 16 via accumulation
args = SFTConfig(
    output_dir=f"{OUT_DIR}/student",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS,
    gradient_accumulation_steps=16 // BS,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=20,
    logging_steps=10, eval_strategy="steps", eval_steps=50, save_strategy="epoch",
    bf16=True, gradient_checkpointing=True,
    max_length=3072,          # older TRL (<0.13): rename to max_seq_length
    packing=False, report_to="none",
)
trainer = SFTTrainer(model=model, args=args, peft_config=peft_cfg,
                     train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tok)
trainer.train()

In [ ]:
# ---- the distilled student on the same held-out set -----------------------
trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True
tuned_out = generate_batch(trainer.model, [s["report"] for s in eval_run])
tuned_scores = score(eval_run, tuned_out)
print("distilled student:", {k: f"{v:.2f}" for k, v in tuned_scores.items()})

In [ ]:
# ---- base vs distilled ----------------------------------------------------
labels = list(base_scores)
bx = np.arange(len(labels))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(bx - w / 2, [base_scores[k] for k in labels], w,
       label=f"Base {STUDENT_MODEL.split('/')[-1]}", color=PAL[0])
ax.bar(bx + w / 2, [tuned_scores[k] for k in labels], w,
       label="Distilled student", color=PAL[1])
ax.set_xticks(bx, labels)
ax.set_ylim(0, 1)
ax.set_ylabel("accuracy on held-out ground truth")
ax.set_title("What did the distillation buy?", loc="left")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

comparison = pd.DataFrame({"base": base_scores, "distilled": tuned_scores}).T.round(3)
print(comparison)
print("\nteacher accuracy on their own pool (for reference):")
print(stats)

In [ ]:
# ---- save the expert ------------------------------------------------------
final_dir = f"{OUT_DIR}/vib-expert-{STUDENT_MODEL.split('/')[-1]}-lora"
trainer.model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
print("LoRA adapter saved to", final_dir)

# Optional: copy to Drive so it survives the runtime
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r {final_dir} /content/drive/MyDrive/

# Optional: merge to a standalone bf16 model for vLLM / GGUF export
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="cpu")
# merged = PeftModel.from_pretrained(base, final_dir).merge_and_unload()
# merged.save_pretrained(f"{OUT_DIR}/vib-expert-merged"); tok.save_pretrained(f"{OUT_DIR}/vib-expert-merged")

# Optional: push to the Hub
# from huggingface_hub import login; login()
# trainer.model.push_to_hub("your-name/vib-expert-lora")

## Where to take this next

- **Close the sim-to-real gap.** The synthetic generator teaches the *method* (speed inference, order matching, envelope reading, ISO grading). Mix in real spectra — CWRU and Paderborn bearing sets, NASA IMS run-to-failure, and this repo's C-MAPSS turbofan data for RUL-style reasoning — by rendering them into the same report format and letting the teachers label them (you lose exact ground truth, so fall back to teacher-consensus voting between GLM-5.2 and Kimi K3 instead of simulator checks).
- **Harden the curriculum.** Add multi-fault samples, resonance/structural cases, variable-speed sweeps, 60 Hz fleets, axial/horizontal channel pairs, and "insufficient data — request more measurements" as a legitimate verdict.
- **Go beyond SFT.** With accepted *and rejected* traces you already have preference pairs for DPO; or run on-policy distillation (GKD in TRL) against a locally-served teacher.
- **Deploy.** Merge the adapter and serve with vLLM, or export GGUF for edge boxes next to the machines.

**Caveats**: severity zone boundaries here are the simplified ISO 20816-3 group 1/2 tables; real programs use machine-specific baselines and trend alarms. Synthetic accuracy ≠ field accuracy — treat the eval numbers as a measure of *distillation transfer*, not of production readiness.